# CareLoom: First Support Agent — Larkspur Electronics

The triage graph from the last notebook decided *what kind of* question came in. This
notebook builds the first version of CareLoom that can actually *do* something about it:
a small ReAct-style agent, running on Groq, with two tools a support agent reaches for
constantly —

- **web search**, for anything that changes day to day (courier delays, holiday shipping
  cutoffs, exchange rates for a cross-border refund)
- **a calculator**, for the arithmetic that comes with almost every refund or discount
  conversation

LangGraph short-term memory is thread-scoped and persisted through checkpoints, and Tavily integration requires `TAVILY_API_KEY` in the environment. The Deep Agents docs also show the general pattern of adding tools to an agent.

## Learning goals

By the end of this notebook, you should be able to:

1. Load `GROQ_API_KEY` and `TAVILY_API_KEY` from `.env`.
2. Create a calculator tool.
3. Connect a Tavily web search tool.
4. Build a simple tool-using agent.
5. Run the agent with a `thread_id` so memory is preserved.


## 1) Install packages


In [ ]:
%pip install -qU langgraph langchain langchain-core langchain-groq langchain-tavily python-dotenv


## 2) Load API keys from `.env`

Create a `.env` file like this:

```env
GROQ_API_KEY=your_groq_api_key
TAVILY_API_KEY=your_tavily_api_key
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=careloom-first-agent
```

Tavily's LangChain integration docs say to set `TAVILY_API_KEY` in the environment. Short-term memory docs say thread-scoped state is managed through checkpoints.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "true").lower() == "true"
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "careloom-first-agent")

print("GROQ_API_KEY set:", bool(GROQ_API_KEY))
print("TAVILY_API_KEY set:", bool(TAVILY_API_KEY))
print("LANGSMITH_API_KEY set:", bool(LANGSMITH_API_KEY))
print("LANGSMITH_TRACING:", LANGSMITH_TRACING)
print("LANGSMITH_PROJECT:", LANGSMITH_PROJECT)


## 3) Connect the Tavily search tool

CareLoom uses Tavily for anything time-sensitive that a static knowledge base can't answer well — courier delays, holiday shipping cutoffs, current exchange rates for a cross-border refund.

The Tavily wrapper validates `TAVILY_API_KEY` as soon as you construct it, not just when you call it — so a missing key fails immediately, before the agent is even built. The cell below tries the current `langchain-tavily` package, falls back to the older `langchain-community` class name if needed, and if neither can find a working key, falls back again to a small offline mock tool with the same calling interface, so the rest of the notebook can still run.


In [ ]:
from langchain_core.tools import tool


def _build_mock_search_tool():
    @tool
    def web_search(query: str) -> str:
        """Search the web for current information, such as exchange rates or courier delays."""
        return (
            f'[offline-fallback] No live TAVILY_API_KEY / network reachable for query: "{query}". '
            "Set TAVILY_API_KEY and run with network access to get real search results."
        )
    return web_search


try:
    from langchain_tavily import TavilySearch
    search_tool = TavilySearch(max_results=2)
    search_tool.name = "web_search"
    search_tool.description = "Search the web for current information, such as exchange rates or courier delays."
except Exception as e:
    print(f"[offline-fallback] langchain-tavily unavailable ({type(e).__name__}: {e}).")
    try:
        from langchain_community.tools.tavily_search import TavilySearchResults
        search_tool = TavilySearchResults(max_results=2)
    except Exception as e2:
        print(f"[offline-fallback] langchain-community Tavily fallback also unavailable ({type(e2).__name__}: {e2}).")
        print("[offline-fallback] Using an offline mock search tool so the agent can still be built.")
        search_tool = _build_mock_search_tool()

search_tool


## 4) Define a calculator tool

Almost every refund or discount conversation needs arithmetic — a percentage off, a currency conversion, a partial refund. We take an expression string rather than fixed arguments so the agent can hand it whatever it computed, not just two plain integers.


In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, such as "49.99 * 83.1" or "6499 * 0.15"."""
    safe_globals = {"__builtins__": {}}
    try:
        value = eval(expression, safe_globals, {})
        return str(round(value, 2))
    except Exception as e:
        return f"Calculator error: {e}"

calculator


## 5) Create the Groq model

The notebook uses Groq for the agent model. You can set `GROQ_MODEL` in `.env` if you want to change it later.


In [ ]:
import os
from langchain_groq import ChatGroq

GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
    max_retries=3,                                 # network/429/5xx retries
    model_kwargs={"parallel_tool_calls": False},   # one tool call per turn
)
print(llm)


## 6) Build the agent

The agent uses the tools and keeps thread-scoped memory with `InMemorySaver`.

LangGraph short-term memory is scoped to a thread and stored through checkpoints, which is why a `thread_id` matters when you invoke the agent.


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

tools = [search_tool, calculator]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are CareLoom, a support assistant for Larkspur Electronics. "
        "Use web search for anything time-sensitive (exchange rates, courier delays, "
        "holiday shipping cutoffs) and the calculator for arithmetic like discounts "
        "and refund amounts. Keep answers short and concrete."
    ),
    checkpointer=InMemorySaver(),
)
print("Agent ready.")


## 7) Run the sample request

A real support question that needs both tools: a live exchange rate (web search) and a currency conversion (calculator).

`invoke_with_retry` handles two different failure modes, and it's worth telling them apart:

- **`tool_use_failed`** — Groq occasionally mis-formats a tool call. This is a generation glitch, not a problem with the input, and it usually succeeds on retry. `ChatGroq`'s own `max_retries` does not cover this case (that only retries network/429/5xx errors, not this kind of 400).
- **Everything else** — a missing API key, no network route to Groq or Tavily, and so on. Retrying won't fix that, so instead of retrying we return a clearly-labeled placeholder and keep going. That's the path you'll see if you run this notebook without live keys; with real `GROQ_API_KEY` and `TAVILY_API_KEY` values and network access, you'll get a genuine model answer instead.


In [ ]:
import time
from langchain_core.messages import HumanMessage, AIMessage

def invoke_with_retry(agent, messages, config, max_attempts=3, base_delay=1.5):
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            return agent.invoke({"messages": messages}, config=config)
        except Exception as e:
            if "tool_use_failed" in str(e):
                last_error = e
                print(f"[retry {attempt}/{max_attempts}] tool_use_failed, retrying...")
                time.sleep(base_delay * attempt)
                continue
            # Not a formatting glitch -- a missing key, unreachable host, etc.
            # Retrying won't help, so fall back to a clearly-labeled placeholder
            # instead of crashing the rest of the notebook.
            print(f"[offline-fallback] Live agent call failed ({type(e).__name__}: {e}).")
            return {
                "messages": messages + [
                    AIMessage(
                        content=(
                            "[offline-fallback] No live Groq/Tavily connection in this environment. "
                            "With real GROQ_API_KEY and TAVILY_API_KEY values and network access, "
                            "this would return a real answer covering the current USD/INR rate and "
                            "the converted refund amount."
                        )
                    )
                ]
            }
    raise last_error

config = {"configurable": {"thread_id": "1"}}

response = invoke_with_retry(
    agent,
    [HumanMessage(content="What's today's USD to INR exchange rate, and what would a $49.99 international return refund be in rupees?")],
    config,
)

print(response["messages"][-1].content)


## 8) Ask a follow-up in the same thread

Because the same `thread_id` is reused, the conversation stays in the same checkpointed thread.


In [ ]:
follow_up = invoke_with_retry(
    agent,
    [HumanMessage(content="And what was that refund amount again?")],
    config,
)

print(follow_up["messages"][-1].content)


## 9) Start a new thread

A new `thread_id` starts a separate conversation while still using the same agent and tools. A different customer's question shouldn't see anything from the thread above.


In [ ]:
config_2 = {"configurable": {"thread_id": "2"}}

new_thread = invoke_with_retry(
    agent,
    [HumanMessage(content="What's 15% off ₹6,499?")],
    config_2,
)

print(new_thread["messages"][-1].content)


## 10) What this notebook demonstrates

- a ReAct-style tool-using agent
- live web search for time-sensitive information
- calculator usage for discount and refund math
- short-term memory through checkpointed threads
- a clean `.env` setup for `GROQ_API_KEY` and `TAVILY_API_KEY`

## Key takeaways

This notebook follows the LangGraph short-term memory model, where conversation state is thread-scoped and persisted through checkpoints, and it uses a simple agent-with-tools pattern similar to the LangChain and Deep Agents examples. Tavily's integration expects `TAVILY_API_KEY`, so putting the key in `.env` keeps the setup clean.

### What's verified where

| Piece | Status |
|---|---|
| Graph structure, `create_agent` wiring, `InMemorySaver` checkpointing | Executed and verified in this sandbox |
| `invoke_with_retry` retry logic for `tool_use_failed` | Code path verified; the retry itself only triggers with a live Groq connection |
| Actual Tavily search results and Groq-generated answers | **Not available in this sandbox** (no outbound network to Groq/Tavily here) — every run above took the `[offline-fallback]` path. Confirm on your own machine with real `GROQ_API_KEY` / `TAVILY_API_KEY` values. |

## References

- Deep research tool setup: https://docs.langchain.com/oss/python/deepagents/deep-research#add-tools
- Short-term memory: https://docs.langchain.com/oss/python/langgraph/short-term-memory
- Tavily integration: https://docs.langchain.com/oss/python/integrations/providers/tavily
